## Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model

import xarray as xr

import numpy as np


## Load Data

In [2]:
is_local_data = False
Month_idx = 4
safe = True
start = None
end = None
max_depth = 3

In [3]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)


In [4]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=4)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=4)
tas_ds = model.shape_data.prune_group_ds_timespan(tas_ds,Month_idx=Month_idx)
pr_ds = model.shape_data.prune_group_ds_timespan(pr_ds,Month_idx=Month_idx)

raw_input_for_test = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")


### some functions to shape the data


In [5]:
def shape_target(ds):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    return ds

In [6]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth").isel(depth=slice(0, max_depth))
    return ds,chunk_mask, detail_mask

In [7]:
def shape_input(ds, chunk_mask):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

### Linear Regression of the mean

In [8]:
mean_target_noneT = shape_target(raw_mrsol_for_mean)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Log_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [9]:
mean_predictors = shape_input(raw_input_for_mean,chunk_mask)

In [10]:
LinReg_mean = model.stats._parallel_linear_regression.ParLinearRegression()

In [11]:
LinReg_mean.fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

In [12]:
var_target_noneT = shape_target(raw_mrsol_for_var)
var_target_noneT, chunk_mask_var, detail_mask_var = mask_stack_target(var_target_noneT)
var_target = model.transform.Log_Transform_ds(var_target_noneT)



In [13]:
var_predictors = shape_input(raw_input_for_var,chunk_mask)

In [14]:
raw_residuals = LinReg_mean.residuals(var_predictors, var_target)

In [15]:
variance_estimator_set = {}

variance_estimator_set["quad_res"] = (raw_residuals.residuals)**2

variance_estimator_set["log_quad_res"] = np.log((raw_residuals.residuals)**2)


### Linear Regression of the Variance

In [16]:
Regr_set_var = {}

In [17]:
for var_key, var_est in variance_estimator_set.items():
    Regr_set_var[var_key] = model.stats._parallel_linear_regression.ParLinearRegression()
    Regr_set_var[var_key].fit(predictors=var_predictors, target=var_est,location_dim="gridcell", regr_dim="time")

In [18]:
if safe:
    for var_key, var_est in variance_estimator_set.items():
        model.save.save_params(LinReg_mean.params,Regr_set_var[var_key].params,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/variance_transform/{var_key}_log_transform/local{is_local_data}/month{Month_idx}", name=f"start={start},end={end}.max_depth{max_depth}")